# 09 - Modelado con SARIMAX

Ajusta modelos SARIMAX individuales por embalse para la predicción del porcentaje de llenado a 7, 30 y 90 días, en los dos escenarios del diseño experimental.


## 1. Configuración

In [1]:
import sys
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX

sys.path.append("..")

DIR_PROCESSED = Path("../data/processed")
DIR_FIGURAS = Path("../outputs/figures")
DIR_RESULTADOS = Path("../outputs/tables")
DIR_RESULTADOS.mkdir(parents=True, exist_ok=True)

TRAIN_INI = pd.Timestamp("2006-01-01")
TEST_INI = pd.Timestamp("2022-01-01")
TEST_FIN = pd.Timestamp("2023-12-31")

HORIZONTES = [7, 30, 90]
PARAMETROS = ["amonio_mgl", "conductividad_uscm", "oxigeno_mgl",
              "ph", "temp_agua_c", "turbidez_ntu"]

df = pd.read_parquet(DIR_PROCESSED / "dataset_modelado.parquet")
experimentales = sorted(df["ID_SAIH"].unique())

print(f"Observaciones: {len(df):,} | embalses: {len(experimentales)}")
print(f"Periodo: {df['fecha'].min():%Y-%m-%d} a {df['fecha'].max():%Y-%m-%d}")

Observaciones: 111,758 | embalses: 17
Periodo: 2006-01-01 a 2023-12-31


## 2. Definición de los escenarios

Las variables exógenas se agrupan en dos conjuntos que difieren únicamente en la incorporación del bloque de calidad del agua. Los términos armónicos que representan el ciclo anual se incluyen en ambos.

In [2]:
EXOG_BASE = [
    "aportacion_m3s",
    "aemet_precipitacion_mm", "prec_acum7", "prec_acum30",
    "aemet_temp_media_c", "et0_tipificada",
    "dia_anio_sin", "dia_anio_cos",
]

EXOG_CALIDAD = [f"cal_{p}" for p in PARAMETROS]

ESCENARIOS = {
    "base": EXOG_BASE,
    "base_calidad": EXOG_BASE + EXOG_CALIDAD,
}

for nombre, cols in ESCENARIOS.items():
    print(f"{nombre}: {len(cols)} variables exógenas")
    faltan = [c for c in cols if c not in df.columns]
    if faltan:
        print(f"  AVISO: no están en el dataset -> {faltan}")

base: 8 variables exógenas
base_calidad: 14 variables exógenas


## 3. Preparación de las series

Para cada embalse se construye la serie objetivo y la matriz de variables exógenas sobre una rejilla diaria continua.


In [3]:
DETERMINISTAS = ["dia_anio_sin", "dia_anio_cos"]

def preparar_serie(df, embalse, exog_cols, horizonte, indice_comun=None, test_ini=TEST_INI):
    """Serie objetivo y matriz de exógenas de un embalse, sobre rejilla diaria continua.

    Las variables observadas se rezagan el horizonte de predicción. Si se pasa
    indice_comun, la serie se restringe exactamente a esas fechas, de modo que los tres
    escenarios (base, base+calidad y persistencia) compartan la misma muestra. No se
    imputa: las filas con exógenas ausentes (por el desplazamiento, las ventanas de los
    acumulados o el inicio del muestreo de calidad) quedan fuera de la muestra."""
    g = (df[df["ID_SAIH"] == embalse].sort_values("fecha")
         .set_index("fecha").asfreq("D"))

    y = g["pct_llenado"]
    exog = pd.DataFrame(index=g.index)
    for c in exog_cols:
        s = g[c] if c in DETERMINISTAS else g[c].shift(horizonte)
        exog[c] = s

    if indice_comun is not None:
        return y.loc[indice_comun], exog.loc[indice_comun]

    valido = y.notna() & exog.notna().all(axis=1)
    if not valido.any():
        return y.iloc[:0], exog.iloc[:0]
    idx = valido[valido].index
    return y.loc[idx], exog.loc[idx]

def indice_valido_comun(df, embalse, horizonte):
    """Fechas con datos completos en el escenario más exigente (base+calidad).
    Define la muestra común de los tres escenarios. Se verifica su continuidad,
    requisito para el ajuste temporal del SARIMAX."""
    y, _ = preparar_serie(df, embalse, ESCENARIOS["base_calidad"], horizonte)
    idx = y.index
    if len(idx) > 0:
        rango = pd.date_range(idx.min(), idx.max(), freq="D")
        if len(idx) != len(rango):
            print(f"  AVISO {embalse} h{horizonte}: índice común con "
                  f"{len(rango) - len(idx)} huecos internos (SARIMAX requiere continuidad)")
    return idx

# Comprobación sobre un embalse
idx = indice_valido_comun(df, experimentales[0], 30)
y, exog = preparar_serie(df, experimentales[0], ESCENARIOS["base_calidad"], 30, indice_comun=idx)
print(f"Embalse {experimentales[0]}: {len(y):,} observaciones válidas (muestra común)")
print(f"Periodo: {y.index.min():%Y-%m-%d} a {y.index.max():%Y-%m-%d}")
print(f"Exógenas: {exog.shape[1]}")

Embalse E002: 6,515 observaciones válidas (muestra común)
Periodo: 2006-03-01 a 2023-12-31
Exógenas: 14


## 4. Métricas y funciones de evaluación

Se define el conjunto de métricas y las tres funciones de evaluación: SARIMAX con origen móvil, persistencia como referencia, y el cálculo de métricas común a ambas.

In [4]:
def metricas(y_real, y_pred):
    """RMSE, MAE y coeficiente de eficiencia de Nash-Sutcliffe."""
    y_real, y_pred = np.asarray(y_real), np.asarray(y_pred)
    m = ~(np.isnan(y_real) | np.isnan(y_pred))
    y_real, y_pred = y_real[m], y_pred[m]
    if len(y_real) == 0:
        return {"n": 0, "rmse": np.nan, "mae": np.nan, "nse": np.nan}
    err = y_real - y_pred
    sst = ((y_real - y_real.mean()) ** 2).sum()
    return {
        "n": len(y_real),
        "rmse": np.sqrt((err ** 2).mean()),
        "mae": np.abs(err).mean(),
        "nse": 1 - (err ** 2).sum() / sst if sst > 0 else np.nan,
    }

In [5]:
def evaluar_sarimax(y, exog, horizonte, orden=(1, 1, 1), test_ini=TEST_INI):
    """Estima los parámetros sobre el periodo de entrenamiento y evalúa con origen móvil.

    Entre el fin del entrenamiento y el inicio del test se deja un embargo igual al
    horizonte. Los parámetros estimados con entrenamiento se aplican a la serie completa
    sin reestimación. En cada origen t del periodo de test se genera una previsión
    dinámica a h pasos: el modelo emplea las observaciones disponibles hasta t y propaga
    con valores predichos hasta t+h."""
    fin_train = test_ini - pd.Timedelta(days=horizonte)
    y_tr, exog_tr = y[y.index < fin_train], exog[exog.index < fin_train]

    if len(y_tr) < 730 or len(y[y.index >= test_ini]) < horizonte + 30:
        return None, None

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        ajuste = SARIMAX(y_tr, exog=exog_tr, order=orden,
                         enforce_stationarity=False,
                         enforce_invertibility=False).fit(disp=False)
        estado = ajuste.apply(y, exog=exog, refit=False)

    idx = y.index
    pos = np.where(idx >= test_ini)[0]
    origenes = pos[pos + horizonte < len(idx)]

    pred = []
    for i in origenes:
        p = estado.get_prediction(start=i + 1, end=i + horizonte, dynamic=True)
        pred.append(p.predicted_mean.iloc[-1])

    return ajuste, pd.DataFrame({
        "fecha": idx[origenes + horizonte],
        "real": y.iloc[origenes + horizonte].values,
        "pred": pred,
    })

In [6]:
def evaluar_persistencia(y, horizonte, test_ini=TEST_INI):
    """Predicción persistente: el valor observado en el origen se mantiene h días.
    Constituye la referencia mínima que el modelo debe superar."""
    idx = y.index
    pos = np.where(idx >= test_ini)[0]
    origenes = pos[pos + horizonte < len(idx)]
    return pd.DataFrame({
        "fecha": idx[origenes + horizonte],
        "real": y.iloc[origenes + horizonte].values,
        "pred": y.iloc[origenes].values,
    })

## 5. Ejecución

Se ajustan y evalúan los modelos para los 17 embalses, los tres horizontes y los dos escenarios, junto con la referencia de persistencia. Los resultados se guardan para su análisis y para la redacción de la memoria.

In [7]:
resultados = []
predicciones = []

for h in HORIZONTES:
    for emb in experimentales:
        # Muestra común: fechas válidas en el escenario más exigente (base+calidad).
        # Los tres escenarios (persistencia, base, base+calidad) se evalúan sobre ellas.
        idx_comun = indice_valido_comun(df, emb, h)
        if len(idx_comun) == 0:
            print(f"  {emb} h{h}: sin datos válidos")
            continue

        # Baseline de persistencia, sobre la muestra común
        y_ref, _ = preparar_serie(df, emb, ESCENARIOS["base_calidad"], h, indice_comun=idx_comun)
        res_p = evaluar_persistencia(y_ref, h)
        m = metricas(res_p["real"], res_p["pred"])
        resultados.append({"embalse": emb, "horizonte": h, "escenario": "persistencia", **m})

        # SARIMAX en cada escenario, todos sobre la muestra común
        for nombre, cols in ESCENARIOS.items():
            try:
                y, exog = preparar_serie(df, emb, cols, h, indice_comun=idx_comun)
                _, res = evaluar_sarimax(y, exog, h)
                if res is None:
                    print(f"  {emb} h{h} {nombre}: datos insuficientes")
                    continue
                m = metricas(res["real"], res["pred"])
                resultados.append({"embalse": emb, "horizonte": h, "escenario": nombre, **m})
                predicciones.append(res.assign(embalse=emb, horizonte=h, escenario=nombre))
            except Exception as e:
                print(f"  ERROR en {emb} h{h} {nombre}: {e}")
                continue
        print(f"  {emb} h{h} listo")

resultados = pd.DataFrame(resultados)
resultados.to_parquet(DIR_RESULTADOS / "sarimax_metricas.parquet", index=False)
pd.concat(predicciones).to_parquet(DIR_RESULTADOS / "sarimax_predicciones.parquet", index=False)

print("\n=== Mediana por horizonte y escenario ===")
print(resultados.groupby(["horizonte", "escenario"])[["rmse", "mae", "nse"]].median().round(3).to_string())

  E002 h7 listo
  E008 h7 listo
  E009 h7 listo
  E011 h7 listo
  E025 h7 listo
  E026 h7 listo
  E027 h7 listo
  E028 h7 listo
  E029 h7 listo
  E030 h7 listo
  E031 h7 listo
  E033 h7 listo
  E07A h7 listo
  E32A h7 listo
  E35A h7 listo
  E570 h7 listo
  E571 h7 listo
  E002 h30 listo
  E008 h30 listo
  E009 h30 listo
  E011 h30 listo
  E025 h30 listo
  E026 h30 listo
  E027 h30 listo
  E028 h30 listo
  E029 h30 listo
  E030 h30 listo
  E031 h30 listo
  E033 h30 listo
  E07A h30 listo
  E32A h30 listo
  E35A h30 listo
  E570 h30 listo
  E571 h30 listo
  E002 h90 listo
  E008 h90 listo
  E009 h90 listo
  E011 h90 listo
  E025 h90 listo
  E026 h90 listo
  E027 h90 listo
  E028 h90 listo
  E029 h90 listo
  E030 h90 listo
  E031 h90 listo
  E033 h90 listo
  E07A h90 listo
  E32A h90 listo
  E35A h90 listo
  E570 h90 listo
  E571 h90 listo

=== Mediana por horizonte y escenario ===
                          rmse    mae    nse
horizonte escenario                         
7         base   

## 6. Resultados

Se examinan las métricas agregadas por horizonte y escenario, el desglose por embalse y la comparación sistemática frente a la persistencia.

In [8]:
tabla = resultados.pivot_table(index="embalse", columns=["horizonte", "escenario"],
                               values="nse").round(3)
print(tabla[30].to_string())

# Cuántos embalses supera SARIMAX a la persistencia
for h in HORIZONTES:
    r = resultados[resultados["horizonte"] == h].pivot_table(
        index="embalse", columns="escenario", values="nse")
    mejora = (r["base"] > r["persistencia"]).sum()
    print(f"h={h}: SARIMAX base supera a persistencia en {mejora}/17 embalses")

escenario   base  base_calidad  persistencia
embalse                                     
E002      -0.459        -0.466        -0.500
E008      -0.309        -0.304        -0.840
E009      -0.285        -0.278        -0.370
E011      -0.076        -0.074        -0.113
E025      -0.037        -0.041        -0.059
E026       0.391         0.389         0.308
E027      -0.611        -0.613        -0.410
E028       0.878         0.878         0.705
E029      -1.255        -1.206        -0.958
E030      -0.274        -0.270        -0.755
E031      -0.046        -0.056        -0.466
E033      -0.313        -0.304        -0.803
E07A       0.819         0.819         0.621
E32A      -0.892        -0.892        -0.780
E35A       0.450         0.452         0.360
E570       0.658         0.660         0.658
E571      -0.407        -0.400        -1.105
h=7: SARIMAX base supera a persistencia en 14/17 embalses
h=30: SARIMAX base supera a persistencia en 14/17 embalses
h=90: SARIMAX base supera a 

## Tablas de resultados para la memoria


In [9]:
# pip install tabulate

In [10]:
import pandas as pd

met = pd.read_parquet("../outputs/tables/sarimax_metricas.parquet")

NOMBRES = {
    "E002": "E002 · Os Peares", "E008": "E008 · Fuente del Azufre", "E009": "E009 · Montearenas",
    "E011": "E011 · Peñarrubia", "E025": "E025 · Leboreiro Mao", "E026": "E026 · Edrada Mao",
    "E027": "E027 · San Esteban", "E028": "E028 · Vilasouto", "E029": "E029 · San Pedro",
    "E030": "E030 · Velle", "E031": "E031 · Castrelo", "E033": "E033 · Frieira",
    "E07A": "E07A · Bárcena", "E32A": "E32A · Albarellos", "E35A": "E35A · Conchas",
    "E570": "E570 · Santiago", "E571": "E571 · Pumares",
}
ESC = {"base": "Base", "base_calidad": "Base + calidad", "persistencia": "Persistencia"}

# --- Tabla 1: medianas por horizonte y escenario ---
t1 = (met.groupby(["horizonte", "escenario"])[["rmse", "mae", "nse"]]
        .median().round(3).reset_index())
t1["escenario"] = t1["escenario"].map(ESC)
t1.columns = ["Horizonte", "Escenario", "RMSE", "MAE", "NSE"]
t1.to_csv("../outputs/tables/sarimax_t1_medianas.csv", index=False)

# --- Tablas 2-4: NSE por embalse en cada horizonte, con fila de mediana ---
def tabla_embalse(h):
    t = (met[met["horizonte"] == h]
         .pivot_table(index="embalse", columns="escenario", values="nse")
         .round(3)[["base", "base_calidad", "persistencia"]])
    t.loc["Mediana"] = t.median().round(3)
    t.index = [NOMBRES.get(e, e) for e in t.index]
    t.columns = [ESC[c] for c in t.columns]
    t.to_csv(f"../outputs/tables/sarimax_nse_h{h}.csv")
    return t

print("TABLA 1 — Medianas por horizonte y escenario")
print(t1.to_markdown(index=False))
for h in [7, 30, 90]:
    print(f"\nTABLA — NSE por embalse (horizonte {h} días)")
    print(tabla_embalse(h).to_markdown())

TABLA 1 — Medianas por horizonte y escenario
|   Horizonte | Escenario      |   RMSE |   MAE |    NSE |
|------------:|:---------------|-------:|------:|-------:|
|           7 | Base           |  4.299 | 2.692 |  0.639 |
|           7 | Base + calidad |  4.289 | 2.714 |  0.638 |
|           7 | Persistencia   |  4.42  | 2.877 |  0.634 |
|          30 | Base           |  7.171 | 4.882 | -0.274 |
|          30 | Base + calidad |  7.189 | 4.849 | -0.27  |
|          30 | Persistencia   |  8.122 | 6.062 | -0.41  |
|          90 | Base           | 11.036 | 9.287 | -0.429 |
|          90 | Base + calidad | 11.057 | 9.303 | -0.516 |
|          90 | Persistencia   | 11.129 | 8.974 | -0.891 |

TABLA — NSE por embalse (horizonte 7 días)
|                          |   Base |   Base + calidad |   Persistencia |
|:-------------------------|-------:|-----------------:|---------------:|
| E002 · Os Peares         |  0.639 |            0.638 |          0.634 |
| E008 · Fuente del Azufre |  0.032 |   